# Bayesian Optimization of the GROMACS Production MDP

**Study:** `md_prod_v1` &nbsp;·&nbsp; **Sampler:** Optuna TPE with `constant_liar=True` &nbsp;·&nbsp; **8 workers × 25 trials = 200 trials**

## Goal

The goal is to find the fastest possible GROMACS production configuration that still passes a stability filter. The aim is not to maximize physical accuracy regardless of computational cost, but to obtain sufficiently reliable and physically stable results as fast as possible. MD production is the computational bottleneck of the MM-GBSA calculation, and not all MDP parameters can be varied independently. Therefore, we used *Bayesian optimization* to identify the fastest compatible parameter set satisfying predefined stability criteria. First, we identified mutually compatible parameter combinations using a single reference system, which led to the definition of **12 valid MDP parameter sets**. The final evaluation was performed on a benchmark set of eight targets with 30 ligands each, providing a broader statistical basis for selecting the final configuration.

## How Bayesian optimization works

BO is a surrogate-model search:
1. **Sample** a configuration from the search space.
2. **Evaluate** — here: run 2 ns of production MD, measure the wall-clock, apply a stability filter.
3. **Update** the internal model of which regions of the space are good.
4. **Suggest** a new configuration with high expected improvement.
5. Repeat.

Optuna's TPE (Tree-structured Parzen Estimator) models P(good | params) and P(bad | params) and picks parameters with a high ratio. `constant_liar=True` lets 8 workers run in parallel — while a trial is in flight the sampler assumes it will return the current mean, which nudges the other workers to explore elsewhere and prevents them from all racing to the same point.


> **Reader guide.** *Experiment A4:* the BO report itself — fitness distribution, top-5
> winners, parameter-importance analysis, and reviewer-iteration notes.
>
> **Question:** *what does the BO fitness landscape look like, and how confident are we in the
> top-5 winners that Experiment B2 will verify at production scale?*
>
> **Method:** Optuna DB read → fitness distribution, per-parameter marginal, top-5 pick.
>
> **Reproducibility contract:** reads `data/external/bayesopt/bayes_opt.db` (repo-relative
> path — no hardcoded /mnt); top-5 winner JSONs at `data/external/bayesopt/bo_winner_v{1..5}.json`.
> Reviewer-iteration companions under `notebooks/bo_ancillary/`.

In [ ]:
# ============================== IDIS house palette (from GPUbench) ==============================
import json
import os
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from matplotlib import cycler

NAVY, GOLD, GREY, GREYD, CREAM, WHITE = "#2F3761", "#8A730F", "#D5D7DF", "#9AA3B5", "#F7F4EF", "#FFFFFF"

plt.rcParams.update({
    "figure.dpi": 110, "figure.facecolor": WHITE, "savefig.facecolor": WHITE,
    "axes.facecolor": WHITE, "axes.edgecolor": NAVY, "axes.labelcolor": NAVY,
    "axes.titlecolor": NAVY, "axes.titleweight": "bold", "axes.grid": False,
    "text.color": NAVY, "xtick.color": NAVY, "ytick.color": NAVY,
    "axes.prop_cycle": cycler(color=[NAVY, GOLD]), "legend.frameon": True,
    "legend.facecolor": WHITE, "legend.edgecolor": GREY, "font.size": 11,
})

DB = Path(os.environ.get("BAYES_OPT_DB", str(Path(__file__).resolve().parents[1] / "data/external/bayesopt/bayes_opt.db") if "__file__" in globals() else "../data/external/bayesopt/bayes_opt.db"))
STUDY = "md_prod_v1"
TRIAL_PS = 2000  # each trial ran 2 ns MD

def ns_per_day(wall_s):
    """Convert a per-trial wall-clock (seconds) to ns/day, using TRIAL_PS."""
    return (TRIAL_PS / 1000.0) * 86400.0 / wall_s

con = sqlite3.connect(DB)


## 1. Search space

12 categorical MDP parameters (discrete choice sets). Only the *production* stage is varied — solvation, energy minimisation, NVT, and NPT are held constant, so every wall-clock difference is attributable to the production MDP alone (this is the BO invariant of the study).

The v1 space is enumerated below directly from the trial-parameter table. **v2 (running, §10)** extends this space with `dt ∈ {3, 4, 5} fs`, `hmr_factor ∈ {2.5, 3.0, 3.5, 4.0}`, `lincs_order ∈ {6, 8}`, `lincs_iter = 2`, and an `mts` toggle — printed alongside the v1 table below.


In [ ]:
params = pd.read_sql_query("""
    SELECT DISTINCT tp.param_name, tp.distribution_json
    FROM trial_params tp
    JOIN trials t ON tp.trial_id = t.trial_id
    JOIN studies s ON t.study_id = s.study_id
    WHERE s.study_name = ?
""", con, params=(STUDY,))

search_space = []
for _, row in params.iterrows():
    dist = json.loads(row['distribution_json'])
    choices = dist['attributes']['choices']
    search_space.append({
        'parameter': row['param_name'],
        'n_choices': len(choices),
        'choices': ', '.join(str(c) for c in choices),
    })

space_df = pd.DataFrame(search_space).sort_values('parameter').reset_index(drop=True)
n_combos = int(space_df['n_choices'].prod())
print(f"Search-space size: {n_combos:,} combinations across {len(space_df)} parameters")
print(f"We sampled 200/{n_combos:,} = {200/n_combos*100:.2f}% of the space.\n")
space_df

# --- v2 extensions ---
try:
    from pipeline_mps.bayes_opt.space_v2 import (
        DT_CHOICES, HMR_CHOICES, LINCS_ORDER_CHOICES, LINCS_ITER_CHOICES, MTS_FACTOR_CHOICES,
    )
    v2_axes = pd.DataFrame([
        {"axis": "dt (ps)",            "v1 pool": [0.001, 0.002],  "v2 pool": DT_CHOICES},
        {"axis": "hmr_factor",         "v1 pool": [1.0],           "v2 pool": HMR_CHOICES},
        {"axis": "lincs_order",        "v1 pool": [4],             "v2 pool": LINCS_ORDER_CHOICES},
        {"axis": "lincs_iter",         "v1 pool": [1],             "v2 pool": LINCS_ITER_CHOICES},
        {"axis": "mts",                "v1 pool": [False],         "v2 pool": [False, True]},
        {"axis": "mts_level2_factor",  "v1 pool": ["N/A"],         "v2 pool": MTS_FACTOR_CHOICES},
    ])
    print("v2 search-space extensions (relative to v1):")
    print(v2_axes.to_string(index=False))
except Exception as exc:
    print(f"(space_v2 module not importable here: {exc})")


## 2. Load the raw trials

Optuna records, per trial: the parameter values, the objective value (fitness), start/end timestamps, and a `cfg` user attribute holding the fully expanded MDP.

**Fitness = 1 / md_production_wall_hours** — higher is faster. Trials that fail the stability filter are recorded as `fitness = 0.0`.


In [ ]:
trials = pd.read_sql_query("""
    SELECT t.number, tv.value AS fitness,
           t.datetime_start, t.datetime_complete
    FROM trials t
    JOIN trial_values tv ON t.trial_id = tv.trial_id
    JOIN studies s ON t.study_id = s.study_id
    WHERE s.study_name = ?
    ORDER BY t.number
""", con, params=(STUDY,))

trials['datetime_start'] = pd.to_datetime(trials['datetime_start'])
trials['datetime_complete'] = pd.to_datetime(trials['datetime_complete'])
trials['wall_s'] = (trials['datetime_complete'] - trials['datetime_start']).dt.total_seconds()
trials['md_wall_s'] = np.where(trials['fitness'] > 0, 3600 / trials['fitness'], np.nan)
# ns_per_day derives from md_wall_s; NaN propagates through arithmetic
trials['ns_per_day'] = ns_per_day(trials['md_wall_s'])
trials['stable'] = trials['fitness'] > 0

print(f"Total trials: {len(trials)}")
print(f"Stable:       {trials['stable'].sum()}  ({trials['stable'].mean()*100:.0f}%)")
print(f"Filtered out: {(~trials['stable']).sum()}")
print(f"Best fitness: {trials['fitness'].max():.3f}")
print(f"Fastest run:  {trials.loc[trials['fitness'].idxmax(), 'ns_per_day']:.0f} ns/day")
trials.head()

## 3. Convergence

The standard BO diagnostic: **best-so-far** should climb monotonically, and the scatter shows the exploration/exploitation balance. TPE samples broadly early on, then clusters around the optimum region — the crossing trial where best-so-far reaches 90% of its final value is annotated on the plot below.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

stable = trials[trials["stable"]]
unstable = trials[~trials["stable"]]

# Convergence: fitness per trial + best-so-far
best_so_far = trials["fitness"].cummax()
final_max = float(best_so_far.iloc[-1])
crossing_idx = int(np.searchsorted(best_so_far.values, 0.9 * final_max))
crossing_trial = crossing_idx + 1  # 1-indexed
ax1.scatter(stable["number"], stable["fitness"], s=25, alpha=0.75,
            c=NAVY, label=f"stable (n={len(stable)})")
ax1.scatter(unstable["number"], unstable["fitness"], s=30, alpha=0.75,
            c=GREYD, marker="x", label=f"unstable (n={len(unstable)})")
ax1.plot(trials["number"], best_so_far, color=GOLD, lw=2, label="best so far")
ax1.axvline(crossing_trial, color=NAVY, lw=1, ls=":", alpha=0.6)
ax1.text(0.02, 0.96,
         f"90% of max @ trial {crossing_trial}",
         transform=ax1.transAxes, fontsize=8, color=NAVY,
         va="top", bbox=dict(boxstyle="round,pad=0.3", fc=WHITE, ec=GREY, lw=0.6))
ax1.set_xlabel("trial")
ax1.set_ylabel("fitness [h$^{-1}$]")
ax1.set_title("Convergence over 200 trials")
ax1.legend(loc="lower right")
ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)

# Throughput histogram
winner_ns = float(stable["ns_per_day"].max())
median_ns = float(stable["ns_per_day"].median())
ax2.hist(stable["ns_per_day"], bins=30, color=NAVY, alpha=0.85, edgecolor=WHITE)
ax2.axvline(winner_ns, color=GOLD, lw=2,
            label=f"winner: {winner_ns:.0f} ns/day")
ax2.axvline(median_ns, color=NAVY, lw=1.5, ls=":",
            label=f"median: {median_ns:.0f} ns/day")
ax2.annotate(f"{winner_ns:.0f}", xy=(winner_ns, 0), xycoords=("data", "axes fraction"),
             xytext=(4, 4), textcoords="offset points",
             color=GOLD, fontsize=9, weight="bold")
ax2.set_xlabel("ns/day (extrapolated from 2 ns wall)")
ax2.set_ylabel("count")
ax2.set_title("Throughput distribution (stable trials)")
ax2.legend(loc="upper left")
ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()
print(f"90% of best-so-far reached at trial {crossing_trial} ({crossing_trial/len(trials)*100:.0f}% of budget)")
unstable_pct = 100 * len(unstable) / len(trials)
print(f"unstable trial fraction: {unstable_pct:.0f}%")


**Interpretation:** the best-so-far reaches 90% of its final value by the trial index printed above. The unstable fraction (also printed above) is dominated by LINCS warnings, temperature drift beyond ±5 K, and density drift above 5% over the window. These thresholds catch integrator blow-ups (a ±5 K drift or 5% density change on a solvated protein means the system exploded) — not fine ensemble-quality issues.


## 4. Decode parameter values per trial

Optuna stores categorical parameters as float indices — we resolve each index back to the actual choice using the distribution stored alongside the trial.


In [ ]:
raw = pd.read_sql_query(
    """
    SELECT t.number, tp.param_name, tp.param_value, tp.distribution_json
    FROM trials t
    JOIN trial_params tp ON tp.trial_id = t.trial_id
    JOIN studies s ON t.study_id = s.study_id
    WHERE s.study_name = ?
    """,
    con, params=(STUDY,),
)
# Cache one choices list per parameter (12 parses instead of ~2400)
choices_by_param = {
    n: json.loads(g.iloc[0])["attributes"]["choices"]
    for n, g in raw.groupby("param_name")["distribution_json"]
}
assert (raw["param_value"] % 1 == 0).all(), "param_value has non-integer entries — categorical index expected"
raw["value"] = [choices_by_param[n][int(v)]
                for n, v in zip(raw["param_name"], raw["param_value"])]
params_wide = raw.pivot(index="number", columns="param_name", values="value").reset_index()
df = trials.merge(params_wide, on="number")
df.head()


## 5. Parameter effects

For every parameter: distribution of fitness per value. **A tight box with a high median = a clear winning value.** Wide, overlapping distributions mean the parameter either doesn't matter or is coupled to another one. The Kruskal–Wallis / ε² per subplot marginalises over the other 11 parameters — TPE's late-stage clustering makes those backgrounds unbalanced, so read the significance as *effect visible in the sampler's exploited region*, not as an unbiased main effect.


In [ ]:
# scipy.stats hoisted to cell 1

params_to_plot = ["dt", "constraints", "integrator", "rcut", "fourierspacing",
                  "nstlist", "tcoupl", "pcoupl", "tau_p", "tau_t",
                  "nstpcouple", "nstcalcenergy"]

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
df_stable = df[df["stable"]].copy()
MIN_N_PER_BOX = 5

# First pass: collect raw p-values and cached medians so we don't recompute
subplot_data = []
for param in params_to_plot:
    if param not in df_stable.columns:
        subplot_data.append(None); continue
    grouped = list(df_stable.groupby(param)["ns_per_day"])
    grouped = [(str(k), v.values) for k, v in grouped if len(v) >= MIN_N_PER_BOX]
    if len(grouped) < 2:
        subplot_data.append(None); continue
    medians = {k: float(np.median(v)) for k, v in grouped}
    grouped.sort(key=lambda kv: -medians[kv[0]])
    try:
        H, p = stats.kruskal(*(v for _, v in grouped))
    except ValueError:
        H, p = float("nan"), float("nan")
    k = len(grouped); n = sum(len(v) for _, v in grouped)
    eps2 = float(max(0.0, (H - k + 1) / (n - k))) if n > k and not np.isnan(H) else float("nan")
    subplot_data.append((param, grouped, H, p, eps2))

# Holm-Bonferroni adjustment across valid (non-NaN) K-W p-values only
raw_p = [d[3] for d in subplot_data if d is not None]
valid_idx = [i for i, p in enumerate(raw_p) if not (p != p)]  # skip NaN
m = len(params_to_plot)   # pre-specified family of tests, not the n>=5 survivors
adj_p = [None] * len(raw_p)
if m:
    order = sorted(valid_idx, key=lambda i: raw_p[i])
    running_max = 0.0
    for rank, i in enumerate(order):
        adj = min(1.0, raw_p[i] * (m - rank))
        running_max = max(running_max, adj)
        adj_p[i] = running_max

idx = 0
for ax, entry in zip(axes.flat, subplot_data):
    if entry is None:
        ax.set_visible(False); continue
    param, grouped, H, p, eps2 = entry
    labels, data = zip(*grouped)
    bp = ax.boxplot(data, patch_artist=True, widths=0.6,
                    medianprops=dict(color=GOLD, lw=1.5),
                    whiskerprops=dict(color=NAVY),
                    capprops=dict(color=NAVY),
                    flierprops=dict(marker=".", markerfacecolor=GREYD,
                                    markeredgecolor=GREYD, markersize=4))
    ax.set_xticks(range(1, len(labels)+1))
    ax.set_xticklabels(labels)
    for i, patch in enumerate(bp["boxes"]):
        patch.set_facecolor(GOLD if i == 0 else NAVY)
        patch.set_alpha(0.75 if i == 0 else 0.55)
        patch.set_edgecolor(NAVY)
    p_str = f"p_adj={adj_p[idx]:.1e}" if adj_p[idx] is not None else "p_adj=N/A"
    ax.set_title(f"{param}  ·  H={H:.1f}, ε²={eps2:.2f}, {p_str}", fontsize=9)
    ax.tick_params(axis="x", rotation=20, labelsize=9)
    ax.set_ylabel("ns/day", fontsize=8)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    idx += 1

plt.suptitle("Throughput per parameter value (ns/day) (stable trials, sorted by median; highest-median group GOLD; p_adj = Holm–Bonferroni over 12 tests)",
             fontsize=11, y=1.01, color=NAVY, weight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()


**Observations:**
- **`dt = 0.002 ps`** wins by construction (half the step count at 1 fs); BO confirms no stability penalty at 2 fs with `h-bonds`.
- **`constraints = h-bonds`** freezes all X–H stretches (~10 fs period) that would otherwise cap dt at 1 fs; the 2 fs ceiling is then set by fast H-bearing angle librations (water HOH, methyl), which is why `h-angles` unlocks `dt ≥ 4 fs` (see §10).
- **`nstlist = 40`** dominates — larger neighbour lists amortize the rebuild cost.
- **`integrator = md`** (leap-frog) beats `md-vv` because `md-vv` incurs an extra force pass under coupled thermo/barostats; `sd`'s slowdown is real per-step work from stochastic collisions.
- **`tcoupl = berendsen`** wins on wall-clock but does not sample the canonical ensemble; swap in `v-rescale` if a canonical distribution is required — the boxplot quantifies the cost. The same caveat applies if `pcoupl = Berendsen` wins: it does not sample NPT and biases density.
- **`fourierspacing = 0.14 nm`** cuts FFT cost; the 0.9 nm real-space cutoff keeps the Ewald balance under `ewald_rtol=1e-5`, and `DispCorr = EnerPres` (fixed) compensates the LJ tail truncation at `rvdw = rcut`.
- **`rcut = 0.9 nm`** — shorter cutoff means fewer pair interactions.


## 6. Top-10 configurations


In [ ]:
top10 = df_stable.nlargest(10, 'fitness')[[
    'number', 'fitness', 'ns_per_day',
    'dt', 'constraints', 'integrator', 'rcut', 'fourierspacing',
    'nstlist', 'tcoupl', 'pcoupl', 'tau_p', 'tau_t',
]].reset_index(drop=True)
top10['fitness'] = top10['fitness'].round(3)
top10['ns_per_day'] = top10['ns_per_day'].round(0).astype(int)
top10

top10_fit = top10["fitness"].astype(float)
core_keys = ["dt", "constraints", "integrator", "rcut", "fourierspacing", "nstlist", "pcoupl", "tcoupl"]
n_unique_core = top10[core_keys].drop_duplicates().shape[0]
log_spread = float(np.log(top10_fit).std())
print(f"Top-10 fitness  mean={top10_fit.mean():.3f}  std={top10_fit.std():.3f}")
print(f"Top-10 log-fitness std (fractional wall-time spread): {log_spread*100:.1f}%")
print("(Stage-2 20 ns re-verify will provide the actual noise band around the winner.)")
print(f"Unique core-parameter tuples in top-10: {n_unique_core}")


**Convergence signal:** the top 10 agree on `dt`, `constraints`, `integrator`, `rcut`, `fourierspacing`, `nstlist`, `pcoupl`, and `tcoupl` — only the weak time-constants (`tau_p`, `tau_t`, `nstpcouple`) vary. The fitness spread across the top 10 is quantified below (std / mean, and count of unique configurations on the plateau).


## 7. Wall-time analysis


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
df_ok = df[df["stable"]].sort_values("number")
wall_min = df_ok["md_wall_s"].min()
wall_med = df_ok["md_wall_s"].median()

ax.scatter(df_ok["number"], df_ok["md_wall_s"], s=20, alpha=0.65, c=GREYD, label="trial")
ax.plot(df_ok["number"], df_ok["md_wall_s"].cummin(), color=NAVY, lw=1.8, label="best so far")
ax.axhline(wall_min, color=GOLD, lw=2, ls="--",
           label=f"fastest: {wall_min:.0f} s / 2 ns")
ax.axhline(wall_med, color=NAVY, lw=1, ls=":",
           label=f"median: {wall_med:.0f} s / 2 ns")
ax.set_xlabel("trial index")
ax.set_ylabel("md_production wall-clock [s]")
ax.set_title(f"MD wall-time per trial  ·  {wall_med/wall_min:.1f}\u00d7 spread median-to-fastest")
ax.legend(loc="upper right", fontsize=9, ncol=2)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
plt.tight_layout(); plt.show()

def _fmt_row(label, wall_s):
    ns_per_day_val = ns_per_day(wall_s)
    hours_for_20ns = 20.0 * 24.0 / ns_per_day_val
    return f"{label:8s} 2 ns in {wall_s:6.0f} s  ->  {ns_per_day_val:5.0f} ns/day  ->  {hours_for_20ns:5.2f} h for a 20 ns replica"

print(_fmt_row("Winner:", wall_min))
print(_fmt_row("Median:", wall_med))
print(_fmt_row("Slowest:", df_ok["md_wall_s"].max()))


## 8. Winner configuration

The full MDP of the best trial — plugged directly into every signac statepoint of the discovery-9 production run as `mdp_override` (720 jobs total).


In [ ]:
winner_row = df_stable.nlargest(1, "fitness").iloc[0]
winner_no = int(winner_row["number"])
w_fit = winner_row["fitness"]
w_nsd = winner_row["ns_per_day"]

cfg_json = pd.read_sql_query("""
    SELECT tua.value_json FROM trial_user_attributes tua
    JOIN trials t ON tua.trial_id=t.trial_id
    JOIN studies s ON t.study_id=s.study_id
    WHERE s.study_name=? AND t.number=? AND tua.key='cfg'
""", con, params=(STUDY, winner_no))
assert not cfg_json.empty, f"no cfg for trial {winner_no}"
winner_cfg = json.loads(cfg_json.iloc[0]["value_json"])
mdp_dict = winner_cfg.get("mdp", {}) or {}
ff_dict = winner_cfg.get("forcefield", {}) or {}

print(f"Winner: trial #{winner_no}  fitness = {w_fit:.3f} h^-1  ({w_nsd:.0f} ns/day)")
print()
print("=== MDP (production) ===")
for k, v in mdp_dict.items():
    print(f"  {k:32s} = {v}")
print()
print("=== Force field ===")
for k, v in ff_dict.items():
    print(f"  {k:32s} = {v}")


## 9. Downstream status

**Stage 2 verify (queued):** the top 5 configurations from v1 are re-verified at **20 ns** instead of 2 ns. A 2 ns window may be too short for the stability filter — a configuration that looks stable at 2 ns can still drift thermally over a long run. Stage 2 filters those out. Job `9484931` is waiting for a GPU slot.

**Discovery-9 production (prepared):** the winner MDP is exported to `configs/bo_winner_v1.json` and wired into every signac statepoint of the 720-job production workspace (`scripts/init_discovery9_winner.py` + `scripts/submit_array_discovery9.sbatch`, 8 targets \u00d7 30 ligands \u00d7 3 replicas). Submission uses gpupack MPS packing, which self-calibrates the pack depth per GPU from `pipeline-mps/calibration.yaml`, so no manual `LIGANDS_PER_TASK` tuning is needed.

**Expected wall-time for discovery-9** (v1 winner, single-GPU 40 ns/day):
- Serial: 720 \u00d7 30 min = 360 GPU-hours \u2192 ~18 h wall on 20 A100s.
- gpupack-packed: ~1/3 of the serial wall assuming the standing calibration transfers to the v1 winner MDP.


## 10. Beyond v1 \u2014 higher dt via HMR and LINCS-iter (v2 study, running)

v1 capped `dt` at 2 fs because our seed topology triggered grompp errors on hydrogen-mass-repartitioning and multiple-time-stepping. Re-running those checks with GROMACS 2025.4 on the ff14SB seed (`scripts/hmr_mts_smoke.sbatch`, job `9496393`) showed the errors are gone and 4 of 5 extended configurations run cleanly:

| config | dt | HMR | LINCS (order / iter) | 200-step perf | ratio vs v1 |
|---|---|---|---|---|---|
| v1 winner baseline | 2 fs | \u2014 | 4 / 1 | 82.7 ns/day | 1.00\u00d7 |
| dt=3 fs + HMR=3.024 | 3 fs | 3.024 | 4 / 1 | 413 ns/day | 5.0\u00d7 |
| **dt=4 fs + HMR=3.024 + LINCS 6/2** | **4 fs** | **3.024** | **6 / 2** | **473 ns/day** | **5.7\u00d7** |
| dt=4 fs + all-bonds + LINCS 8/2 | 4 fs | \u2014 | 8 / 2 | 241 ns/day | 2.9\u00d7 |
| dt=5 fs + HMR=4 + MTS | 5 fs | 4.0 | 8 / 2 | grompp fatal (bond period < 5\u00b7dt) | \u2014 |

The 200-step ns/day numbers overstate steady-state throughput because per-run initialization (PME setup, neighbour-list build, DD tuning) dominates; v2 re-measures at the 2 ns budget for like-for-like comparison.

A v2 BO study (`md_prod_v2`) is running now with an extended search space: `dt \u2208 {1, 2, 3, 4, 5} fs`, `hmr \u2208 {1.0, 2.5, 3.0, 3.5, 4.0}`, `lincs_order \u2208 {4, 6, 8}`, `lincs_iter \u2208 {1, 2}`. It is warm-started from the v1 top-5 so TPE starts on a proven plateau and only spends budget exploring the new axes. Job `9499454` (8 workers \u00d7 25 trials).

When v2 completes we replace `configs/bo_winner_v1.json` with `bo_winner_v2.json`, re-run `gpupack calibrate` on the v2 MDP, and launch discovery-9 on the updated recipe. Re-calibration matters because dt = 4 fs roughly halves the per-ns kernel launches, so gpupack's stacking decisions from the v1 calibration are unlikely to transfer.


## 9a. Stage-2 verify (top-5 at 20 ns)

Auto-populates when the `md_prod_v1_verify20ns` study appears in the DB (Stage-2 job re-runs each top-5 v1 trial at 20 ns). Until then this section prints "queued".

In [ ]:
# Stage-2: top-5 at 20 ns — auto-fill when the study exists
def _study_exists(name):
    row = con.execute("SELECT 1 FROM studies WHERE study_name=?", (name,)).fetchone()
    return row is not None

stage2 = "md_prod_v1_verify20ns"
if not _study_exists(stage2):
    print(f"Stage-2 study '{stage2}' not yet in DB.")
    print("Waiting for job 9484931 (top-5 verify at 20 ns) to populate results.")
else:
    s2 = pd.read_sql_query(
        """
        SELECT t.number, tv.value AS verify_fitness
        FROM trials t
        JOIN trial_values tv ON tv.trial_id=t.trial_id
        JOIN studies s ON s.study_id=t.study_id
        WHERE s.study_name=?
        ORDER BY tv.value DESC
        """,
        con, params=(stage2,),
    )
    s2["verify_ns_day"] = np.where(s2["verify_fitness"] > 0,
                                    ns_per_day(3600.0 / s2["verify_fitness"]), np.nan)
    print(f"Stage-2 study loaded: {len(s2)} verify trials\n")
    print(s2.head(10).to_string(index=False))
    if len(s2) >= 3:
        # Bar: v1 stage-1 fitness vs stage-2 verify fitness for the top-5 v1 trials
        top5_v1 = df_stable.nlargest(5, "fitness")[["number", "fitness", "ns_per_day"]]
        merged = top5_v1.merge(s2, on="number", how="left")
        fig, ax = plt.subplots(figsize=(9, 3.5))
        x = np.arange(len(merged))
        ax.bar(x - 0.2, merged["ns_per_day"], width=0.35, color=GREYD, label="Stage-1 (2 ns)")
        ax.bar(x + 0.2, merged["verify_ns_day"], width=0.35, color=NAVY, label="Stage-2 verify (20 ns)")
        ax.set_xticks(x); ax.set_xticklabels([f"trial {n}" for n in merged["number"]])
        ax.set_ylabel("ns/day")
        ax.set_title("Top-5 v1 trials: 2 ns Stage-1 vs 20 ns Stage-2 verify")
        ax.legend(loc="upper right")
        for side in ("top", "right"): ax.spines[side].set_visible(False)
        plt.tight_layout(); plt.show()


## 10a. v2 study — extended dt / HMR / LINCS-iter (auto-fill)

Auto-populates when the `md_prod_v2` study appears in the DB. Compares the v2 top-N against the v1 winner side by side.

In [ ]:
v2_study = "md_prod_v2"
if not _study_exists(v2_study):
    print(f"v2 study '{v2_study}' not yet in DB.")
    print("Waiting for job 9499454 (8 workers x 25 trials, warm-started from v1 top-5) to populate.")
else:
    v2_trials = pd.read_sql_query(
        """
        SELECT t.number, tv.value AS fitness
        FROM trials t
        JOIN trial_values tv ON tv.trial_id=t.trial_id
        JOIN studies s ON s.study_id=t.study_id
        WHERE s.study_name=?
        ORDER BY t.number
        """,
        con, params=(v2_study,),
    )
    v2_trials["ns_per_day"] = np.where(v2_trials["fitness"] > 0,
                                        ns_per_day(3600.0 / v2_trials["fitness"]), np.nan)
    v2_stable_mask = v2_trials["fitness"] > 0
    v2_stable = int(v2_stable_mask.sum())
    v2_best = float(v2_trials["fitness"].max()) if len(v2_trials) else 0.0
    v1_best = float(df_stable["fitness"].max())
    print(f"v2 study: {len(v2_trials)} trials, {v2_stable} stable, best fitness {v2_best:.3f} h^-1")
    print(f"v1 winner: {v1_best:.3f} h^-1  ·  v2 improvement: {v2_best/v1_best:.2f}x")

    # (a) best-so-far v1 vs v2
    if len(v2_trials) >= 5:
        v1_best_so_far = df["fitness"].cummax().to_numpy()
        v2_best_so_far = v2_trials["fitness"].cummax().to_numpy()
        fig, ax = plt.subplots(figsize=(11, 4))
        ax.plot(np.arange(1, len(v1_best_so_far)+1), v1_best_so_far,
                color=GREYD, lw=2, label=f"v1 (best {v1_best:.1f} h$^{{-1}}$)")
        ax.plot(np.arange(1, len(v2_best_so_far)+1), v2_best_so_far,
                color=NAVY, lw=2, label=f"v2 (best {v2_best:.1f} h$^{{-1}}$)")
        ax.set_xlabel("trial"); ax.set_ylabel("best-so-far fitness [h$^{-1}$]")
        ax.set_title(f"v1 vs v2 convergence  ·  v2 gain {v2_best/v1_best:.2f}x")
        ax.legend()
        for side in ("top", "right"): ax.spines[side].set_visible(False)
        plt.tight_layout(); plt.show()

    # (b) v2 parameter-effect boxplots — including the new axes (dt=3/4/5, HMR, LINCS-iter/order, mts)
    if v2_stable >= 20:
        # decode v2 params
        v2_raw = pd.read_sql_query(
            """
            SELECT t.number, tp.param_name, tp.param_value, tp.distribution_json
            FROM trials t
            JOIN trial_params tp ON tp.trial_id=t.trial_id
            JOIN studies s ON s.study_id=t.study_id
            WHERE s.study_name=?
            """,
            con, params=(v2_study,),
        )
        v2_choices = {n: json.loads(g.iloc[0])["attributes"]["choices"]
                      for n, g in v2_raw.groupby("param_name")["distribution_json"]}
        v2_raw["value"] = [v2_choices[n][int(v)]
                           for n, v in zip(v2_raw["param_name"], v2_raw["param_value"])]
        v2_wide = v2_raw.pivot(index="number", columns="param_name", values="value").reset_index()
        v2_df = v2_trials.merge(v2_wide, on="number")
        v2_df_stable = v2_df[v2_df["fitness"] > 0].copy()

        v2_params = ["dt", "hmr_factor", "lincs_order", "lincs_iter", "mts",
                     "constraints", "integrator", "rcut", "fourierspacing", "nstlist",
                     "tcoupl", "pcoupl"]
        v2_params = [p for p in v2_params if p in v2_df_stable.columns]
        n = len(v2_params); ncols = 4; nrows = (n + ncols - 1) // ncols
        fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3.2*nrows))
        axes_flat = axes.flat if hasattr(axes, "flat") else [axes]
        for ax, param in zip(axes_flat, v2_params):
            grouped = list(v2_df_stable.groupby(param)["ns_per_day"])
            grouped = [(str(k), v.values) for k, v in grouped if len(v) >= 3]
            if len(grouped) < 2:
                ax.set_visible(False); continue
            grouped.sort(key=lambda kv: -np.median(kv[1]))
            labels, data = zip(*grouped)
            bp = ax.boxplot(data, patch_artist=True, widths=0.6,
                            medianprops=dict(color=GOLD, lw=1.5),
                            whiskerprops=dict(color=NAVY),
                            capprops=dict(color=NAVY),
                            flierprops=dict(marker=".", markerfacecolor=GREYD,
                                            markeredgecolor=GREYD, markersize=4))
            ax.set_xticks(range(1, len(labels)+1))
            ax.set_xticklabels(labels)
            for i, patch in enumerate(bp["boxes"]):
                patch.set_facecolor(GOLD if i == 0 else NAVY)
                patch.set_alpha(0.75 if i == 0 else 0.55)
                patch.set_edgecolor(NAVY)
            ax.set_title(param, fontsize=9)
            ax.tick_params(axis="x", rotation=20, labelsize=8)
            ax.set_ylabel("ns/day", fontsize=8)
            for side in ("top", "right"): ax.spines[side].set_visible(False)
        # hide unused axes
        for ax in list(axes_flat)[len(v2_params):]:
            ax.set_visible(False)
        plt.suptitle("v2 parameter-effect boxplots (extended search: dt=1..5 fs, HMR, LINCS-order/iter, MTS)",
                     fontsize=11, y=1.005, color=NAVY, weight="bold")
        plt.tight_layout(); plt.show()
    else:
        print(f"(v2 has only {v2_stable} stable trials so far — parameter-effect plot needs >=20 to render)")


## 11. Random-search baseline

The bootstrap-random overlay in §3 is a lower bound because it shuffles TPE's own trial pool. A dedicated `md_prod_v1_random` study (200 trials of the same v1 space with `RandomSampler`, same 2 ns budget) is the proper baseline. Auto-populates when the study appears in the DB.

To launch: `sbatch scripts/bayes_opt_random.sbatch` — 8 workers × 25 trials, same wall envelope as the TPE run.

In [ ]:
rand_study = "md_prod_v1_random"
if not _study_exists(rand_study):
    print(f"Random baseline study '{rand_study}' not yet in DB.")
    print("Launch with: sbatch scripts/bayes_opt_random.sbatch")
else:
    r = pd.read_sql_query(
        """
        SELECT t.number, tv.value AS fitness
        FROM trials t
        JOIN trial_values tv ON tv.trial_id=t.trial_id
        JOIN studies s ON s.study_id=t.study_id
        WHERE s.study_name=?
        ORDER BY t.number
        """,
        con, params=(rand_study,),
    )
    tpe_best_so_far = df["fitness"].cummax().to_numpy()
    rnd_best_so_far = r["fitness"].cummax().to_numpy()
    tpe_final = tpe_best_so_far[-1]
    rnd_final = rnd_best_so_far[-1] if len(rnd_best_so_far) else 0.0

    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(np.arange(1, len(tpe_best_so_far)+1), tpe_best_so_far,
            color=NAVY, lw=2, label=f"TPE (best {tpe_final:.1f} h$^{{-1}}$)")
    ax.plot(np.arange(1, len(rnd_best_so_far)+1), rnd_best_so_far,
            color=GOLD, lw=2, label=f"Random (best {rnd_final:.1f} h$^{{-1}}$)")
    ax.set_xlabel("trial"); ax.set_ylabel("best-so-far fitness [h$^{-1}$]")
    ax.set_title(f"TPE vs Random  ·  same 200-trial budget  ·  gain {tpe_final/max(rnd_final,1e-9):.2f}x")
    ax.legend()
    for side in ("top", "right"): ax.spines[side].set_visible(False)
    plt.tight_layout(); plt.show()
    # Trial-index at which TPE crosses random's final value
    if len(rnd_best_so_far):
        cross = int(np.searchsorted(tpe_best_so_far, rnd_final))
        print(f"TPE reached Random's final best by trial {cross+1} "
              f"({(cross+1)/len(tpe_best_so_far)*100:.0f}% of the budget).")


In [ ]:
con.close()